# Lab 10 — Instruction-Tuning Schemas
### Week 2 · Data Engineering for LLM Pipelines

Lab 09 got you clean JSONL. Now you shape it into the layouts a **fine-tuning** run actually
consumes. You'll take the Lab 09 outputs (RAG chunks + an SFT view — regenerated here so
this notebook stands alone) and produce three schemas, then apply the hygiene that keeps a
training set honest: **dedupe**, **length governance**, and **decontamination** against your
eval set.

**By the end you will be able to:**
1. Contrast the three SFT layouts — **Trio** (`instruction`/`input`/`output`),
   **prompt–completion**, and **chat messages** — and say when each applies.
2. Convert a Lab 09 SFT view and RAG chunks into instruction-tuning JSONL with stable
   metadata and templated prompts.
3. Apply dataset hygiene: **dedupe**, **language/governance** filtering, **min/max length**
   control, and **eval decontamination**.
4. **Validate** each schema (pydantic) and compute basic dataset statistics.

> **Hints stay light (Labs 05–09).** Each Part opens with a **Toolbox**; you assemble the
> pieces. Target **23/23**; a red check never halts the notebook.
>
> ⚠️ **CURRENCY FLAG — the modern SFT schema is chat *messages*.** Current hosted
> fine-tuning (OpenAI and others) expects one conversation per line —
> `{"messages": [{"role":"system",...},{"role":"user",...},{"role":"assistant",...}]}` —
> **not** the legacy `{"prompt","completion"}` shape, and not inline `<system>/<user>`
> tag-strings. We build all three so you can read older datasets, but treat **messages** as
> the target. (Uploading prompt–completion to a chat model is now rejected as the wrong
> format.)


## Setup — imports, folders, and the `check()` helper

In [ ]:
%pip install -r requirements.txt

In [ ]:

import json, re, hashlib, itertools
from pathlib import Path
from typing import Optional, List, Dict, Literal

import numpy as np
import pandas as pd
import orjson
from pydantic import BaseModel, Field, ValidationError, field_validator

for p in ["artifacts/jsonl", "artifacts/samples", "artifacts/stats", "data"]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("ready | pandas", pd.__version__)

In [ ]:

_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    try:
        ok = bool(predicate() if callable(predicate) else predicate); note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'\u2705 PASS' if ok else '\u274c FAIL'} \u2014 {label}{note}")
def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing  ({_score['fail']} to go)\n{'='*46}")

def read_jsonl(path):
    p = Path(path)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []
def write_jsonl(path, records):
    with open(path, "wb") as f:
        for r in records:
            f.write(orjson.dumps(r) + b"\n")
    return len(records)

check("Setup: artifact folders exist", lambda: all(Path(p).is_dir() for p in ["artifacts/jsonl","artifacts/stats"]))


### Provided — regenerate the Lab 09 inputs

In delivery these files come straight from **Lab 09**. We regenerate them here (same seed,
same pipeline) so this notebook runs on its own: **546** RAG chunk lines and **120** SFT
rows.

In [ ]:

def _bootstrap_lab09(seed=42):
    """Condensed Lab 09 pipeline -> artifacts/jsonl/rag_chunks.jsonl (546) + corpus_sft.jsonl (120)."""
    import random, csv
    from datetime import datetime, timedelta
    rng = random.Random(seed)
    TYPES=["help_article","policy","release_note","faq"]; SEC=["Overview","Setup","Troubleshooting","FAQ"]
    TAGS=["billing","security","compliance","sso","api","governance","export","retention","privacy","rate_limits"]
    LANGS=["en","en","en","de","fr"]; CONF=["public","internal"]; now=datetime(2025,2,10)
    boiler=("This article explains how to configure single sign-on with step-by-step instructions. "
            "Use the admin console to enable SAML and verify claim mappings. "
            "Common pitfalls include clock skew and incorrect audience URIs. ")
    rows=[]
    for i in range(1,1001):
        kind=rng.choice(TYPES); doc_id=f"DOC-{i:04d}"
        title={"help_article":f"How to configure SSO (v{rng.randint(1,5)}).",
               "policy":f"Data Retention Policy \u2014 Region {rng.choice(['US','EU','APAC'])}",
               "release_note":f"Release 2025{rng.randint(1,12):02d} \u2014 Key fixes",
               "faq":f"FAQ: {rng.choice(['Exports','Rate Limits','Privacy','Billing'])}"}[kind]
        body=(boiler*rng.randint(1,3))+f"Additional details about {rng.choice(TAGS)} and {rng.choice(TAGS)}. Ref {doc_id}."
        rows.append({"doc_id":doc_id,"type":kind,"title":title,"section":rng.choice(SEC),"body_text":body,
                     "tags":",".join(rng.sample(TAGS,k=rng.randint(2,4))),
                     "source_url":f"https://example.local/{kind}/{doc_id.lower()}",
                     "created_at":(now-timedelta(days=rng.randint(0,240))).strftime('%Y-%m-%d'),
                     "updated_at":(now-timedelta(days=rng.randint(0,30))).strftime('%Y-%m-%d'),
                     "language":rng.choice(LANGS),"confidentiality":rng.choice(CONF)})
    rows[0].update({"confidentiality":"public","language":"en"})
    for j in range(9): rows.append({**rows[0],"doc_id":f"DOC-DUP-{j:02d}"})
    for k in range(5): rows.append({**rows[0],"doc_id":f"DOC-EMPTY-{k:02d}","body_text":""})
    df=pd.DataFrame(rows)
    nz=lambda s: re.sub(r"\s+"," ",str(s or "")).strip()
    for c in ["title","section","body_text"]: df[c]=df[c].map(nz)
    df=df[(df.confidentiality=="public")&(df.language=="en")]
    df=df[df.body_text.str.len()>=30].copy()
    df["ck"]=[hashlib.sha1((nz(t+" "+b)).lower().encode()).hexdigest() for t,b in zip(df.title,df.body_text)]
    df["updated_at"]=pd.to_datetime(df["updated_at"])
    df=df.sort_values(["ck","updated_at"],ascending=[True,False]).drop_duplicates("ck",keep="first")
    def split_chunks(text,max_chars=300,overlap=60):
        text=re.sub(r"\s+"," ",str(text)).strip()
        if not text: return []
        out=[]; i=0
        while i<len(text):
            end=min(i+max_chars,len(text)); out.append(text[i:end])
            if end==len(text): break
            i=max(0,end-overlap)
        return out
    with open("artifacts/jsonl/rag_chunks.jsonl","wb") as f:
        for r in df.itertuples():
            for j,ch in enumerate(split_chunks(r.body_text)):
                f.write(orjson.dumps({"doc_id":r.doc_id,"chunk_id":f"{r.doc_id}-{j:04d}","text":ch,
                    "metadata":{"title":r.title,"section":r.section,"tags":r.tags,"source_url":r.source_url,
                    "language":r.language,"confidentiality":r.confidentiality,"schema_version":"rag-chunk-v1"}})+b"\n")
    sample=df.sample(min(120,len(df)),random_state=7)
    with open("artifacts/jsonl/corpus_sft.jsonl","wb") as f:
        for r in sample.itertuples():
            f.write(orjson.dumps({"input":f"Summarize the key steps from: {r.title} ({r.section}).",
                "output":"Key steps: enable SAML; map claims; verify time sync; check audience URI; review settings.",
                "metadata":{"doc_id":r.doc_id,"type":r.type,"lang":r.language}})+b"\n")

_bootstrap_lab09()
print("rag_chunks.jsonl:", len(read_jsonl("artifacts/jsonl/rag_chunks.jsonl")),
      "| corpus_sft.jsonl:", len(read_jsonl("artifacts/jsonl/corpus_sft.jsonl")))


---
## Part A — Three schemas, one source of truth

| Schema | Shape | Where it shows up |
|---|---|---|
| **Trio** | `{instruction, input, output, metadata}` | Alpaca / FLAN, open-source SFT — separates task from context |
| **Prompt–Completion** | `{prompt, completion, metadata}` | **legacy** hosted FT & many HF datasets — template baked in |
| **Chat messages** | `{messages:[{role,content}…], metadata}` | **modern** hosted FT (OpenAI &c.) — the target format |

The **Trio** keeps task and context separable, so you can re-template later without
rewriting data. The **messages** layout is what current fine-tuning APIs ingest. You'll emit
all three.


### A1 — `to_messages`

Write `to_messages(instruction, output, context="", system="You are a concise technical assistant.")`
returning the modern list: a `system` message, a `user` message whose content is the
instruction (and the context appended as `\n\nContext: {context}` when context is non-empty),
and an `assistant` message holding the output.

> **🧰 Toolbox for Part A** — a list of `{"role": ..., "content": ...}` dicts · roles are
> `system` / `user` / `assistant` · conditionally append context · f-strings.

In [ ]:

def to_messages(instruction, output, context="", system="You are a concise technical assistant."):
    # TODO: return [system, user, assistant] messages; fold context into the user turn when present.
    return []

_demo = to_messages("Summarize the setup steps.", "Enable SAML; map claims.", context="Some doc text.")
_demo

In [ ]:

_m = to_messages("Do X", "Done", context="CTX")
check("A1: three messages in system/user/assistant order",
      lambda: [d["role"] for d in _m] == ["system","user","assistant"])
check("A1: user turn carries instruction + context; assistant carries output",
      lambda: "Do X" in _m[1]["content"] and "CTX" in _m[1]["content"] and _m[2]["content"] == "Done")
check("A1: no context -> user turn is just the instruction",
      lambda: to_messages("Only I", "O")[1]["content"] == "Only I")


---
## Part B — Trio (and legacy prompt–completion) from the SFT view

Standardize the Lab 09 SFT rows into **Trio** records, then render a **legacy**
prompt–completion view from them.

> **🧰 Toolbox for Part B** — `read_jsonl` / `write_jsonl` (provided) ·
> `(obj.get("metadata") or {}).get("lang")` · `re.sub(r"\s+"," ",s).strip()` ·
> a `set` of `(instruction, output)` for dedupe · f-string templates.


### B1 — Build the Trio file

From `artifacts/jsonl/corpus_sft.jsonl`: keep only `lang == "en"`; the Lab 09 `input` **is**
the instruction; set Trio `input=""`; normalize whitespace on `output` and drop outputs under
**20** chars; **dedupe** on `(instruction, output)`. Each record is
`{instruction, input, output, metadata:{doc_id, schema_version:"trio-v1"}}`. Write to
`artifacts/jsonl/instruct_trio.jsonl`.

In [ ]:

def build_trio(src="artifacts/jsonl/corpus_sft.jsonl", out="artifacts/jsonl/instruct_trio.jsonl"):
    # TODO: en only; instruction = source "input"; Trio input=""; normalize output, drop <20 chars;
    #       dedupe on (instruction, output); metadata {doc_id, schema_version:"trio-v1"}; write + return.
    write_jsonl(out, [])
    return []

trio = build_trio()
print("trio rows:", len(trio))

In [ ]:

_trio = read_jsonl("artifacts/jsonl/instruct_trio.jsonl")
check("B1: 61 trio rows after dedupe", lambda: len(_trio) == 61)
check("B1: every row has instruction/input/output + trio-v1 metadata",
      lambda: len(_trio) > 0 and all({"instruction","input","output","metadata"} <= set(r)
              and r["metadata"].get("schema_version") == "trio-v1" for r in _trio))
check("B1: no duplicate (instruction, output) pairs remain",
      lambda: len(_trio) > 0 and len({(r["instruction"], r["output"]) for r in _trio}) == len(_trio))


### B2 — Render the legacy prompt–completion view

Template each Trio row into `{prompt, completion, metadata}` and write
`artifacts/jsonl/instruct_prompt_completion.jsonl`. Use the header template
`"### Instruction:\n{instruction}\n\n### Response:\n"` for the prompt; the completion is the
Trio `output`.

> ⚠️ This is the **legacy** shape — great for reading old datasets, but the header is baked
> in, so you can't re-template later without regenerating. Contrast with A1's messages.

In [ ]:

def render_prompt_completion(trio_path="artifacts/jsonl/instruct_trio.jsonl",
                             out="artifacts/jsonl/instruct_prompt_completion.jsonl"):
    # TODO: header-template each trio row -> {prompt, completion, metadata}; write + return.
    write_jsonl(out, [])
    return []

pc = render_prompt_completion()
print("prompt-completion rows:", len(pc))

In [ ]:

_pc = read_jsonl("artifacts/jsonl/instruct_prompt_completion.jsonl")
check("B2: 61 prompt-completion rows", lambda: len(_pc) == 61)
check("B2: prompt uses the header template; completion == trio output",
      lambda: len(_pc) > 0 and all("### Instruction:" in r["prompt"] and r["prompt"].endswith("### Response:\n")
              for r in _pc))


---
## Part C — Synthesize instruction data from RAG chunks

Turn **knowledge chunks** into weakly-supervised summarization tasks — a common way to
bootstrap an instruction corpus. Two hygiene steps matter here: **length governance** (skip
trivially short chunks) and **dedupe** (the chunks share a boilerplate intro, so many are
near-identical).

> **🧰 Toolbox for Part C** — `read_jsonl` · `len(text)` for a char-length band ·
> a `set` of chunk **text** for stable dedupe (⚠️ **not** Python's `hash()` — it's salted
> per process and non-reproducible) · `to_messages` from A1.


### C1 — Trio from RAG chunks (length band + dedupe)

From `artifacts/jsonl/rag_chunks.jsonl`: keep chunks whose text length is in
`[250, 1200]`; **dedupe on the chunk text** (stable — a `set`/`hashlib`, never `hash()`);
build a Trio with a fixed summarization `instruction`, the chunk as `input`, and a short
templated `output`. Carry `doc_id`, `chunk_id`, `schema_version:"trio-from-rag-v1"` in
metadata. Write `artifacts/jsonl/instruct_trio_from_rag.jsonl`.

In [ ]:

INSTRUCTION = "Summarize the following section in 2-3 sentences focusing on the key steps and caveats."
TARGET = "Key steps: ensure SSO claims are mapped; verify time sync; review settings as noted."

def build_trio_from_rag(src="artifacts/jsonl/rag_chunks.jsonl",
                        out="artifacts/jsonl/instruct_trio_from_rag.jsonl",
                        min_chars=250, max_chars=1200):
    # TODO: length-band the chunk text; dedupe on the TEXT (stable, not hash()); build a Trio
    #       (instruction, input=text, output=TARGET, metadata w/ doc_id, chunk_id, schema_version).
    write_jsonl(out, [])
    return []

trio_rag = build_trio_from_rag()
print("trio-from-rag rows:", len(trio_rag))

In [ ]:

_tr = read_jsonl("artifacts/jsonl/instruct_trio_from_rag.jsonl")
check("C1: 131 rows (546 chunks -> 403 after length -> 131 after dedupe)", lambda: len(_tr) == 131)
check("C1: chunk texts are unique (dedupe held)",
      lambda: len(_tr) > 0 and len({r["input"] for r in _tr}) == len(_tr))
check("C1: metadata carries chunk_id + trio-from-rag-v1",
      lambda: len(_tr) > 0 and all(r["metadata"].get("chunk_id") and
              r["metadata"].get("schema_version") == "trio-from-rag-v1" for r in _tr))


### C2 — Convert to modern chat *messages*

Turn each `trio_from_rag` row into the **messages** schema with `to_messages` (from A1):
instruction as the task, the chunk text as `context`, the target as the assistant turn.
Carry the same metadata. Write `artifacts/jsonl/instruct_chat_from_rag.jsonl`.

In [ ]:

def build_chat_from_rag(src="artifacts/jsonl/instruct_trio_from_rag.jsonl",
                        out="artifacts/jsonl/instruct_chat_from_rag.jsonl"):
    # TODO: each row -> {"messages": to_messages(instruction, output, context=input), "metadata": ...}
    write_jsonl(out, [])
    return []

chat = build_chat_from_rag()
print("chat-messages rows:", len(chat))

In [ ]:

_chat = read_jsonl("artifacts/jsonl/instruct_chat_from_rag.jsonl")
check("C2: 131 chat rows", lambda: len(_chat) == 131)
check("C2: each row is a messages list with system/user/assistant",
      lambda: len(_chat) > 0 and all([m["role"] for m in r["messages"]] == ["system","user","assistant"] for r in _chat))
check("C2: the chunk text rode into the user turn as context",
      lambda: len(_chat) > 0 and all("Context:" in r["messages"][1]["content"] for r in _chat))


---
## Part D — Validate, govern length, decontaminate

Schema validation with **pydantic**, then two hygiene passes: **length governance** and
**decontamination** against an eval holdout.

> **🧰 Toolbox for Part D** — `pydantic` `BaseModel` + `field_validator` ·
> `Model.model_validate(obj)` in a try/except · word-count length proxy ·
> substring membership for decontamination.


### D1 — Validate each schema

Define pydantic models and `validate_file(path, Model)` → `(total, bad)` where `bad` counts
lines that fail `Model.model_validate`. The **`ChatRow`** must enforce the current hosted-FT
rule: **at least one `user` and one `assistant` message**. Validate all three files.

> **🧰 extra** — `class Msg(BaseModel): role: Literal["system","user","assistant"]; content: str`
> · a `@field_validator("messages")` that checks the roles present.

In [ ]:

class Trio(BaseModel):
    instruction: str; input: str; output: str
    metadata: Optional[Dict] = Field(default_factory=dict)

class PC(BaseModel):
    prompt: str; completion: str
    metadata: Optional[Dict] = Field(default_factory=dict)

# TODO: define Msg (role is a Literal of the 3 roles; content str) and ChatRow (messages: List[Msg],
#       metadata) with a @field_validator that requires >=1 user AND >=1 assistant message.

def validate_file(path, Model):
    # TODO: return (total, bad); bad = lines that fail Model.model_validate.
    return (0, 0)

v_trio = validate_file("artifacts/jsonl/instruct_trio.jsonl", Trio)
v_pc   = validate_file("artifacts/jsonl/instruct_prompt_completion.jsonl", PC)
v_chat = (0, 0)
print("trio:", v_trio, "| pc:", v_pc, "| chat:", v_chat)

In [ ]:

def _rejects_system_only():
    try:
        ChatRow.model_validate({"messages": [{"role": "system", "content": "x"}]}); return False
    except ValidationError:
        return True                          # only a real schema rejection counts
def _bad_role_rejected():
    try:
        ChatRow.model_validate({"messages": [{"role": "user", "content": "x"},
                                             {"role": "robot", "content": "y"}]}); return False
    except ValidationError:
        return True

check("D1: all three schemas validate clean (0 bad)",
      lambda: v_trio == (61, 0) and v_pc == (61, 0) and v_chat == (131, 0))
check("D1: ChatRow rejects a system-only conversation", _rejects_system_only)
check("D1: ChatRow rejects an invalid role", _bad_role_rejected)


### D2 — Length governance + eval decontamination

Filter the prompt–completion file. Drop rows whose `prompt` or `completion` exceeds a token
budget (**word-count proxy**: `MAX_PROMPT_TOK=700`, `MAX_COMP_TOK=350`), **and** drop any row
whose prompt overlaps the eval holdout `EVAL`. Write survivors to
`..._cleansed.jsonl` and record `n_cleansed`.

> ⚠️ **Token proxy.** Word count is a stand-in so the lab runs offline; real length
> governance uses the model's tokenizer (`tiktoken` / `transformers`), and words *undercount*
> tokens (~1 word ≈ 1.3 tokens). ⚠️ **Decontamination matters:** the `Data Retention Policy`
> tasks are in our eval holdout, so they must not appear in training.

In [ ]:

def token_proxy(s):
    return max(1, len(s.split()))

MAX_PROMPT_TOK, MAX_COMP_TOK = 700, 350
EVAL = {"Data Retention Policy"}

def cleanse(src="artifacts/jsonl/instruct_prompt_completion.jsonl",
            out="artifacts/jsonl/instruct_prompt_completion_cleansed.jsonl"):
    # TODO: drop rows whose prompt hits an EVAL trigger, or whose prompt/completion exceed the
    #       token-proxy budgets; write survivors + return them.
    write_jsonl(out, [])
    return []

cleansed = cleanse()
n_cleansed = len(cleansed)
print("cleansed rows:", n_cleansed)

In [ ]:

def _drops_overlong():
    return token_proxy("w " * 800) > MAX_PROMPT_TOK

_cl = read_jsonl("artifacts/jsonl/instruct_prompt_completion_cleansed.jsonl")
check("D2: 52 rows survive (9 Data-Retention-Policy rows decontaminated)", lambda: len(_cl) == 52)
check("D2: no surviving prompt contains an eval trigger",
      lambda: len(_cl) > 0 and not any(t in r["prompt"] for r in _cl for t in EVAL))
check("D2: token_proxy correctly flags an over-budget row", _drops_overlong)


### D3 — Dataset statistics

Compute a stats dict — per-file row `counts` and the mean prompt/completion word-length over
the prompt–completion file — and write `artifacts/stats/instruct_stats.json`.

In [ ]:

def dataset_stats(out="artifacts/stats/instruct_stats.json"):
    # TODO: counts per file (trio, pc, chat) + mean prompt/completion word-length over the pc file;
    #       write JSON + return the dict.
    Path(out).write_text(json.dumps({}, indent=2))
    return {}

stats = dataset_stats()
stats

In [ ]:

check("D3: counts are trio=61, pc=61, chat=131",
      lambda: stats.get("counts") == {"trio": 61, "pc": 61, "chat": 131})
check("D3: mean word-lengths computed (> 0) and stats file written",
      lambda: stats.get("prompt_words_mean", 0) > 0 and Path("artifacts/stats/instruct_stats.json").exists())
score()


---
## Wrap-up — answer in this Markdown cell

1. **Schema choice** — for your capstone, Trio vs prompt–completion vs **messages**, and why.
2. **Re-templating** — show a training-time template and explain how the **Trio** lets you
   change it *without* rewriting the dataset (and why prompt–completion doesn't).
3. **Length thresholds** — what budgets did you set, and how would they shift for a smaller
   vs larger context model? Why is word-count only a proxy?
4. **Decontamination** — where in the pipeline does it run, and what happened to the
   `Data Retention Policy` rows here?

**Key takeaways**
- **Messages is the modern target.** Legacy prompt–completion still appears in older/open
  datasets, but current hosted fine-tuning ingests one **conversation per line**; a chat
  model rejects prompt–completion outright.
- **Keep task and context separable.** The **Trio** re-templates for free; a baked-in prompt
  string does not.
- **Dedupe on content, reproducibly.** Shared boilerplate makes RAG chunks near-identical —
  and never key a dedupe on Python's `hash()` (salted per process; non-reproducible).
- **Govern length with the real tokenizer** in production; a word-count proxy is fine for a
  first pass but undercounts tokens.
- **Decontaminate against your eval holdout** *before* training, or your metrics are
  leaking. Overlap with a shared instruction template can silently swallow a whole slice.
